In [ ]:
"""A/B testing.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1DtE03jG4spfA0zmwxzxkNANQzBbXmqGH
"""

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df= pd.read_csv("marketing_AB.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
"""Data Cleaning"""

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
#converted col bool->num
df["converted"]= df["converted"].astype("int")

In [ ]:
"""# KPI Convertion (main KPI)
1=converted, 0=not converted
"""

In [ ]:
# conversion rate per group
conversion_summary = df.groupby('test group')['converted'].mean()
conversion_summary

In [ ]:
"""Conversion rate was selected as the primary KPI to evaluate campaign effectiveness, as it directly measures the proportion of users who completed the desired action."""

In [ ]:
df.groupby('test group').agg({
    'converted': ['mean', 'sum'],
    'total ads': 'mean'
})

In [ ]:
"""On an average 2.6% of users converted from ads while 1.7% converted from psa.

Initial analysis shows a difference in conversion rates between the two groups. To determine whether this difference is statistically significant, we perform hypothesis testing.

Null Hypothesis (H₀):
The conversion rate of the Ad campaign is equal to the conversion rate of the PSA campaign.

**H₀: Pad= Ppsa**


Alternative Hypothesis (H₁):
The conversion rate of the Ad campaign is greater than the conversion rate of the PSA campaign.

**H₁: Pad= Ppsa**
"""

In [ ]:
#Statistical Testing
#Since conversion is binary we are using proportion test insetad of t-test
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
# count conversions
conversions = df.groupby('test group')['converted'].sum()

In [ ]:
# total users
n = df.groupby('test group')['converted'].count()

In [ ]:
# z-test
z_stat, p_value = proportions_ztest(conversions, n)

In [ ]:
print("Z-stat:", z_stat)
print("P-value:", p_value)

In [ ]:
alpha = 0.05

In [ ]:
if p_value < alpha:
    print("Reject H0 → Significant difference")
else:
    print("Fail to reject H0 → No significant difference")

In [ ]:
"""In z-test above we are checking if the difference here is real (statistically significant) or just random.

As p val is way smaller than 0.05 we reject the null hypothesis. The difference is statistically significant:
Ad campaign performs better than PSA

**Key Insights**



1.   The average number of ads seen is similar across both groups, suggesting that the improved performance is due to campaign quality/content, not higher exposure.
2.   Users exposed to the Ad campaign are more likely to convert under similar conditions, highlighting stronger engagement.

**Recommnedations**


1.   Allocate more budget to the Ad campaign, as it demonstrates significantly higher conversion efficiency.
2.   Analyze elements of the Ad campaign (creative, targeting, messaging) and replicate them across other campaigns.
3. Since exposure levels are similar, focus on improving content quality rather than increasing ad frequency.
"""

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
conversion_summary.plot(kind='bar')
plt.title("Conversion Rate by Group")
plt.ylabel("Conversion Rate")
plt.show()

In [ ]:
df.groupby(['most ads day', 'test group'])['converted'].mean().unstack()

In [ ]:
df.groupby(['most ads hour', 'test group'])['converted'].mean().unstack()

In [ ]:
"""**Insights**

1.   Ad group shows higher conversion rate than PSA
2.   Users exposed to more ads tend to convert more
3. Certain days/hours show peak engagement

**Recommnedations**

1. Allocate more budget to ad campaign
2. Optimize ad delivery during high-performing hours
3. Reduce ad fatigue (if too many ads lower conversion)


"""